# Actividad 1: obtener y preparar datos desde Hugging Face

**Dataset:** [`aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026`](https://huggingface.co/datasets/aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026)

Precio de oferta por metro cuadrado de casas y apartamentos en 38 ciudades de Colombia (2026). Licencia CC BY 4.0, autor: Aptología.

| Columna | Descripción |
|---|---|
| `city` | Ciudad |
| `n_si` | Conteo asociado a casas (ver nota en la transformación 1) |
| `n_no` | Conteo asociado a apartamentos (ver nota en la transformación 1) |
| `casa_cop_m2` | Precio por m² de casas, en pesos colombianos |
| `apto_cop_m2` | Precio por m² de apartamentos, en pesos colombianos |
| `diferencia_pct` | Diferencia porcentual de la casa frente al apartamento: `(casa / apto − 1) × 100`. Negativo = la casa es más barata |

Flujo del notebook:

```
HFDatasetLoader → DataFrame → transformaciones → Reportes → reportes/vivienda_colombia.txt
```

- **`src/clases_po/hf_loader.py`**: clase `HFDatasetLoader`, se conecta a Hugging Face, devuelve un DataFrame y guarda los datos originales en `datos/`.
- **`src/clases_po/reportes.py`**: clase `Reportes`, recibe un DataFrame y genera el reporte `.txt`.
- **Este notebook**: coordina ambas clases y aplica las transformaciones.

## 1. Preparar el entorno e importar las clases

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se pueden importar las clases sin instalar el proyecto como paquete.

In [ ]:
import sys  # Módulo del sistema; permite modificar las rutas donde Python busca módulos
from pathlib import Path  # Para manejar rutas de carpetas

ROOT = Path.cwd().resolve()  # Empezamos en la carpeta desde donde se ejecuta el notebook
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():  # Mientras no encontremos src/clases_po...
    ROOT = ROOT.parent  # ...subimos una carpeta

sys.path.insert(0, str(ROOT / "src"))  # Agregamos src/ a las rutas de búsqueda de Python

import pandas as pd  # Librería para trabajar con DataFrames
from clases_po import HFDatasetLoader, Reportes  # Importamos las dos clases desde src/clases_po

pd.set_option("display.float_format", "{:,.2f}".format)  # Muestra los decimales con 2 cifras y separador de miles

## 2. Obtener el DataFrame desde Hugging Face

El dataset tiene un único split, `train`. Con `filename`, los datos originales también se guardan como CSV en `datos/`.

In [ ]:
loader = HFDatasetLoader(  # Creamos el cargador
    dataset_name="aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026",  # Identificador del dataset en Hugging Face
    split="train",  # División a descargar (es la única que tiene)
    filename="datos/vivienda_colombia.csv",  # Copia de los datos originales en CSV
)
df = loader.load()  # Descargamos el dataset y lo obtenemos como DataFrame de pandas
print(loader.info())  # Información básica: dataset, split, filas y columnas
df.head()  # Mostramos las primeras 5 filas para observar los datos (esto NO es una transformación)

In [ ]:
df.info()  # Revisamos tipos de datos y valores no nulos de cada columna (esto tampoco es una transformación)

## 3. Transformaciones del DataFrame

Cada transformación guarda su resultado en un DataFrame nuevo, para poder comparar el antes y el después.

### Transformación 1: renombrar columnas
Usamos nombres en español y más descriptivos.

> **Nota:** la página del dataset no explica qué cuentan `n_si` y `n_no`. Por sus valores, todo indica que son la cantidad de anuncios de casas (`n_si`) y de apartamentos (`n_no`) usados para calcular cada precio. Confírmalo en el archivo `.md` del repositorio del dataset en Hugging Face; si significan otra cosa, cambia los nombres nuevos en esta celda.

In [ ]:
nuevos_nombres = {  # Diccionario: nombre_original -> nombre_nuevo
    "city": "ciudad",  # Nombre de la ciudad
    "n_si": "n_casas",  # Cantidad de anuncios de casas (interpretación, ver nota)
    "n_no": "n_apartamentos",  # Cantidad de anuncios de apartamentos (interpretación, ver nota)
    "casa_cop_m2": "precio_m2_casa",  # Precio por m² de casas (COP)
    "apto_cop_m2": "precio_m2_apto",  # Precio por m² de apartamentos (COP)
}
df_t1 = df.rename(columns=nuevos_nombres)  # rename() aplica el diccionario y devuelve un DataFrame nuevo
print("Columnas antes:  ", list(df.columns))  # Nombres originales
print("Columnas después:", list(df_t1.columns))  # Nombres nuevos

### Transformación 2: expresar los precios en millones de pesos
Valores como `3490168.698772` son difíciles de leer; los pasamos a millones con 2 decimales (`3.49`).

In [ ]:
df_t2 = df_t1.copy()  # Copia para no modificar df_t1
columnas_precio = ["precio_m2_casa", "precio_m2_apto"]  # Columnas que vamos a convertir
df_t2[columnas_precio] = (df_t2[columnas_precio] / 1_000_000).round(2)  # Dividimos entre un millón y redondeamos
df_t2 = df_t2.rename(columns={"precio_m2_casa": "precio_m2_casa_mill",  # Agregamos "_mill" al nombre para
                              "precio_m2_apto": "precio_m2_apto_mill"})  # dejar claro que la unidad es millones de COP
df_t2["diferencia_pct"] = df_t2["diferencia_pct"].round(1)  # Redondeamos la diferencia porcentual a 1 decimal
df_t2.head()  # Vista del resultado

### Transformación 3: crear la columna `mas_barato`
Indica qué tipo de vivienda tiene el m² más barato en cada ciudad, a partir del signo de `diferencia_pct`.

In [ ]:
df_t3 = df_t2.copy()  # Copia para no modificar df_t2
df_t3["mas_barato"] = (df_t3["diferencia_pct"] < 0).map(  # True si la casa es más barata (diferencia negativa)
    {True: "casa", False: "apartamento"}  # Convertimos True/False en una etiqueta de texto
)
print(df_t3["mas_barato"].value_counts())  # Contamos en cuántas ciudades gana cada tipo
df_t3[["ciudad", "diferencia_pct", "mas_barato"]].head()  # Vista de las columnas relevantes

### Transformación 4: crear la columna `total_anuncios`
Suma de anuncios de casas y apartamentos por ciudad; indica cuántos datos respaldan cada precio.

In [ ]:
df_t4 = df_t3.assign(  # assign() agrega columnas nuevas y devuelve un DataFrame nuevo
    total_anuncios=df_t3["n_casas"] + df_t3["n_apartamentos"]  # Suma fila por fila de ambas columnas
)
df_t4[["ciudad", "n_casas", "n_apartamentos", "total_anuncios"]].head()  # Vista del resultado

### Transformación 5: filtrar ciudades con al menos 1.000 anuncios
Con pocos anuncios el precio promedio es menos confiable, así que dejamos solo las ciudades con una muestra amplia.

In [ ]:
MIN_ANUNCIOS = 1000  # Umbral mínimo de anuncios por ciudad
df_t5 = df_t4[df_t4["total_anuncios"] >= MIN_ANUNCIOS]  # Filtro booleano: conserva solo las filas que cumplen
excluidas = df_t4.loc[df_t4["total_anuncios"] < MIN_ANUNCIOS, "ciudad"].tolist()  # Ciudades que quedaron fuera
print("Ciudades antes del filtro:  ", len(df_t4))  # Cantidad de filas antes
print("Ciudades después del filtro:", len(df_t5))  # Cantidad de filas después
print("Ciudades excluidas:", excluidas)  # Lista de las ciudades eliminadas

### Transformación 6: ordenar por `diferencia_pct`
De mayor descuento de la casa frente al apartamento (más negativo) a menor.

In [ ]:
df_final = df_t5.sort_values("diferencia_pct").reset_index(drop=True)  # Ordena ascendente y renumera el índice desde 0
df_final  # Mostramos el DataFrame final completo

## 4. Generar el reporte .txt con el DataFrame transformado

Se pasa `df_final` (no `df`), para que el reporte describa los datos transformados.

In [ ]:
reportes = Reportes()  # Creamos el objeto de reportes
titulo = "REPORTE: CASA VS APARTAMENTO, PRECIO M2 COLOMBIA 2026 (TRANSFORMADO)"  # Título del reporte
ruta = reportes.guardar_reporte(df_final, "reportes/vivienda_colombia.txt", titulo)  # Guardamos el .txt en reportes/
print(f"Reporte guardado en: {ruta}")  # Confirmamos dónde quedó

## 5. Mostrar el resultado

In [ ]:
reportes.imprimir_reporte(df_final, titulo)  # Imprimimos el mismo reporte que quedó en el archivo

## 6. Explicación de las transformaciones

| # | Transformación | Operación de pandas | Efecto |
|---|---|---|---|
| 1 | Renombrar columnas | `rename(columns=...)` | Nombres en español y descriptivos |
| 2 | Precios en millones de COP | división + `round()` | `3490168.69` → `3.49`; diferencia con 1 decimal |
| 3 | Nueva columna `mas_barato` | comparación + `map()` | Etiqueta `casa` o `apartamento` por ciudad |
| 4 | Nueva columna `total_anuncios` | `assign(...)` | Suma de anuncios de casas y apartamentos |
| 5 | Filtrar ciudades | filtro booleano `df[cond]` | Solo ciudades con ≥ 1.000 anuncios |
| 6 | Ordenar | `sort_values(...)` | De mayor a menor descuento de la casa |

El archivo `reportes/vivienda_colombia.txt` describe **el DataFrame final (`df_final`)**, no el original.

**Fuente de datos:** Aptología, *Casa vs apartamento: precio por m² en ciudades de Colombia 2026*, Hugging Face, licencia CC BY 4.0.